# Job Category Chatbot

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/blob/main/unknown_jobs.ipynb)

My app helps a job board manager decide which country and sector to target for a first category-repair campaign by comparing missing tags and AI-labeled job titles across eight countries.
1. Where do job postings have the most missing categories?
2. How often did Gemini agree with comparable tags, and which sectors were hidden?
3. Which sectors lead after repair, and where should the manager run a pilot? This is my original sub-question.

## 1. Load the data and choose a chat mode
The JSON file contains counts from all 4,000 original postings and the published full-run Gemini results. Its country counts distinguish the source tags `unknown` and `other-general-jobs`.
Run all with `RUN_LIVE = False` for the fast topic chatbot. For fresh Gemini labels and free-text answers, enable your `GOOGLE_API_KEY` in Colab Secrets and set `RUN_LIVE = True`.

In [ ]:
import pandas as pd
import requests
RUN_LIVE = False
url = "https://raw.githubusercontent.com/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/main/project_counts.json"
data = requests.get(url).json()
audit = pd.DataFrame(data["audit"])
hidden = pd.DataFrame(data["hidden"])
print("Postings in the sample:", sum(audit["postings"]))

Postings in the sample: 4000


## 2. Q1 · Find the country with the highest missing share
Add the two missing-tag counts, divide by that country’s postings and multiply by 100. Each country has 500 postings, so the largest share also gives the largest count.

In [ ]:
def missing_by_country(table):
    table["missing"] = table["unknown"] + table["other_general"]
    table["percent"] = 100 * table["missing"] / table["postings"]
    for i, country in enumerate(table["country"]):
        if table["percent"][i] == max(table["percent"]):
            return country

priority = missing_by_country(audit)
audit_answer = f"Highest missing share: {priority}.\n{audit}"
print(audit_answer)

Highest missing share: Germany.
          country  postings  unknown  other_general  missing  percent
0         Germany       500      411             23      434     86.8
1     Switzerland       500       13            342      355     71.0
2          France       500      164            161      325     65.0
3           Spain       500      254             69      323     64.6
4     Netherlands       500      253             28      281     56.2
5           Italy       500       13            146      159     31.8
6  United Kingdom       500        0              2        2      0.4
7             USA       500        0              0        0      0.0


## 3. Create the AI sector column
The full run classified multilingual titles into 16 sectors. The short example uses an originally missing UK title, “Teaching Assistant.” Its recorded Gemini answer is “teaching & education.”

In [ ]:
if RUN_LIVE:
    from google import genai
    from google.colab import userdata
    client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

def ask_gemini(prompt):
    reply = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt)
    return reply.text.strip()

In [ ]:
def label_title(title):
    choices = ", ".join(hidden["sector"])
    prompt = f"UK job title: {title}. Give one of these sectors or uncertain: {choices}"
    label = ask_gemini(prompt).lower()
    if label in data["hidden"]["sector"]:
        return label
    return "uncertain"

demo = pd.DataFrame({"title": ["Teaching Assistant"], "ai_type": ["teaching & education"]})
if RUN_LIVE:
    demo["ai_type"] = demo["title"].apply(label_title)
print(demo)

                title               ai_type
0  Teaching Assistant  teaching & education


## 4. Q2 · Check agreement and find hidden sectors
Of 2,121 postings with known tags, 28 were outside the sector mapping and two had uncertain AI labels. Agreement uses the remaining 2,091 comparisons. It measures consistency with existing tags.

In [ ]:
def agreement_and_hidden(label):
    matches = data["agreement"]["matches"]
    checked = data["agreement"]["checked"]
    rate = round(100 * matches / checked, 1)
    agreement = f"Agreement: {matches}/{checked} ({rate}%). This compares Gemini with existing tags."
    coverage = f"Missing categories filled: {sum(hidden['count'])}/{sum(audit['missing'])}."
    return {"agreement": agreement, "hidden": f"{coverage}\n{hidden}\nUK title check: Teaching Assistant → {label}"}

checks = agreement_and_hidden(demo["ai_type"][0])
print("\n".join(checks.values()))

Agreement: 1355/2091 (64.8%). This compares Gemini with existing tags.
Missing categories filled: 1876/1879.
                              sector  count
0               trade & construction    407
1   logistics, driver & supply chain    227
2                        engineering    204
3      accounting, banking & finance    176
4       sales & business development    157
5               healthcare & nursing    132
6   maintenance, cleaning & facility    122
7          it & software development    102
8                other / specialized     97
9             hospitality & catering     63
10   administrative & office support     52
11        customer service & support     44
12                  hr & recruitment     27
13                social work & care     27
14              teaching & education     24
15       marketing, pr & advertising     15
UK title check: Teaching Assistant → teaching & education


## 5. Q3 · Compare sectors and choose a pilot
The published ranking lists each country’s top three sectors by final count, with ties in alphabetical order. Full-run counts keep known sectors and add valid AI labels for missing ones.

In [ ]:
def sector_shifts(rankings, country, label):
    lines = []
    for name, counts in rankings.items():
        table = pd.DataFrame(counts)
        table["gain"] = table["after"] - table["before"]
        lines.append(f"{name}\n{table}")
    uk = rankings["United Kingdom"]
    before = uk["before"][0]
    gain = int(label == uk["sector"][0])
    lines.append(f"UK one-title scenario: {uk['sector'][0]} {before} → {before + gain}.")
    top = rankings[country]["sector"][0]
    pilot = f"Pilot category repair in {country} and test a campaign for {top} after label review."
    return {"pilot": pilot, "changes": "\n".join(lines)}

In [ ]:
shifts = sector_shifts(data["rankings"], priority, demo["ai_type"][0])
print("\n".join(shifts.values()))

Pilot category repair in Germany and test a campaign for engineering after label review.
France
                          sector  before  after  gain
0           healthcare & nursing     128    156    28
1  accounting, banking & finance       3    105   102
2           trade & construction       0     61    61
Germany
                 sector  before  after  gain
0           engineering       6     81    75
1  healthcare & nursing       9     73    64
2  trade & construction       0     67    67
Italy
                             sector  before  after  gain
0               other / specialized     117    123     6
1  logistics, driver & supply chain      44     52     8
2      sales & business development      12     52    40
Netherlands
                             sector  before  after  gain
0  logistics, driver & supply chain      50    118    68
1              trade & construction       7     63    56
2               other / specialized      35     52    17
Spain
                    

## 6. Ask the chatbot
In the last cell, edit the quoted question and run that cell again. Try “How often did Gemini agree?”, “What sectors were hidden?”, or “Where should we run the pilot?” Turn on live mode in section 1 for open-ended questions about these results.

In [ ]:
class JobChatbot:
    def __init__(self, answers):
        self.answers = answers

    def ask(self, question):
        if RUN_LIVE:
            facts = "\n".join(self.answers.values())
            prompt = f"Use only these results. Say when an answer is unavailable.\n{facts}\nQuestion: {question}"
            return ask_gemini(prompt)
        for keywords, answer in self.answers.items():
            for word in keywords.split(" "):
                if word in question.lower():
                    return answer
        return "Try asking about missing tags, agreement, hidden sectors, changes, or the pilot."

In [ ]:
answers = {"pilot recommend campaign": shifts["pilot"],
           "agree match accuracy q2": checks["agreement"],
           "change shift rank gain leading q3": shifts["changes"],
           "hidden filled recovered": checks["hidden"],
           "missing uncategor audit q1": audit_answer}
chatbot = JobChatbot(answers)

In [ ]:
question = "What is the country with the most missing categories?"
print(chatbot.ask(question))

Highest missing share: Germany.
          country  postings  unknown  other_general  missing  percent
0         Germany       500      411             23      434     86.8
1     Switzerland       500       13            342      355     71.0
2          France       500      164            161      325     65.0
3           Spain       500      254             69      323     64.6
4     Netherlands       500      253             28      281     56.2
5           Italy       500       13            146      159     31.8
6  United Kingdom       500        0              2        2      0.4
7             USA       500        0              0        0      0.0
